# 🥚 Pelatihan Model YOLOv26 Nano Instance Segmentation (`yolo26n-seg`)
### Sistem Inspeksi Mutu Telur SNI 3926:2023 - State-of-the-Art (SOTA) End-to-End

Notebook ini dirancang khusus untuk melatih model generasi terbaru **YOLOv26-seg** menggunakan dataset `Egg Dataset Clean` untuk sistem sortir konveyor dan studi komparasi terhadap YOLOv8-seg & RetinaNet.

---

### 🚀 Keunggulan Arsitektur YOLOv26:
1. **NMS-Free End-to-End**: Meniadakan tahap Non-Maximum Suppression (NMS) sehingga latensi inferensi konstan tanpa *frame-drop* di konveyor.
2. **STAL (Small-Target-Aware Assignment)**: Optimal mendeteksi retakan mikro rambut cangkang (*hairline cracks*) dan bintik noda mikro.
3. **MuSGD Optimizer & ProgLoss**: Konvergensi pelatihan lebih cepat dan stabil pada arsitektur Nano.
4. **Instance Segmentation (Masking)**: Menghasilkan mask poligon noda untuk kalkulasi rasio noda SNI (Mutu I <= 5%, Mutu II <= 10%).

---

### Output yang Diunduh Otomatis:
- **`best.pt`** : Bobot model terbaik YOLOv26-seg.
- **`yolo26_evaluation_graphs.zip`** : Paket kurva performa (Loss, Box/Mask PR curve, F1 curve, Confusion Matrix, val sample).


In [ ]:
# 1. Cek GPU Akselerasi & Install Ultralytics Versi Terbaru
!nvidia-smi
!pip install -q -U ultralytics matplotlib seaborn pillow


In [ ]:
# 2. Ekstrak Dataset (Unggah Egg_Dataset_Clean.zip ke panel files di sebelah kiri)
import os
import shutil
import zipfile
import glob

# Cari file zip di /content
zip_candidates = [
    '/content/Egg_Dataset_Clean.zip',
    '/content/egg_dataset_clean.zip',
    '/content/dataset.zip'
] + glob.glob('/content/*.zip')

zip_path = None
for z in zip_candidates:
    if os.path.exists(z) and 'evaluation_graphs' not in z:
        zip_path = z
        break

extract_dir = '/content/dataset'

if zip_path and os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f'📦 Ditemukan file dataset: {zip_path} ({size_mb:.2f} MB)')
    
    print('Mengekstrak dataset...')
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
        
    # Pastikan data.yaml menggunakan path absolut Colab
    data_yaml_path = os.path.join(extract_dir, 'data.yaml')
    yaml_content = """path: /content/dataset
train: train/images
val: valid/images
test: test/images

nc: 2
names: ['cracked', 'dirt']
"""
    with open(data_yaml_path, 'w', encoding='utf-8') as f:
        f.write(yaml_content)

    train_imgs = len(glob.glob(os.path.join(extract_dir, 'train', 'images', '**', '*.*'), recursive=True))
    val_imgs = len(glob.glob(os.path.join(extract_dir, 'valid', 'images', '**', '*.*'), recursive=True))
    test_imgs = len(glob.glob(os.path.join(extract_dir, 'test', 'images', '**', '*.*'), recursive=True))
    print(f'✅ Dataset siap di {extract_dir}:')
    print(f'   - Train images: {train_imgs}')
    print(f'   - Valid images: {val_imgs}')
    print(f'   - Test images : {test_imgs}')
else:
    print('⚠️ File zip belum ditemukan! Silakan upload Egg_Dataset_Clean.zip ke panel Files di sebelah kiri.')


In [ ]:
# 3. Inisialisasi Model YOLOv26-seg (Nano) Pre-trained
from ultralytics import YOLO

# Memuat model YOLOv26 Nano Instance Segmentation
model_name = "yolo26n-seg.pt"
print(f"Memuat bobot pre-trained: {model_name}...")
model = YOLO(model_name)
print("✅ Model YOLOv26-seg siap dilatih!")


In [ ]:
# 4. Pelatihan Model YOLOv26-seg (80 Epochs pada GPU Colab)
epochs = 80
batch_size = 16
imgsz = 640

print('=' * 75)
print(f'🚀 MEMULAI PELATIHAN YOLOV26-SEG ({epochs} EPOCHS) PADA GPU COLAB')
print('=' * 75)

results = model.train(
    data="/content/dataset/data.yaml",
    epochs=epochs,
    imgsz=imgsz,
    batch=batch_size,
    device=0,
    workers=4,
    project="/content/runs/segment",
    name="egg_yolo26n_seg",
    save=True,
    plots=True,
    exist_ok=True
)

print('\n' + '=' * 75)
print('✅ PELATIHAN SELESAI!')
print('=' * 75)


In [ ]:
# 5. Evaluasi Independen pada Test Split (90 Citra Uji)
print('=' * 75)
print('📊 EVALUASI MODEL PADA TEST SPLIT TELUR')
print('=' * 75)

metrics = model.val(
    data="/content/dataset/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/content/runs/segment",
    name="egg_yolo26n_seg_test",
    exist_ok=True
)

print('\n' + '=' * 75)
print('🏆 METRIK AKHIR YOLOV26-SEG:')
print('=' * 75)
print(f'Mask mAP@0.50     : {metrics.seg.map50 * 100:.2f}%')
print(f'Mask mAP@0.50:0.95: {metrics.seg.map * 100:.2f}%')
print(f'Box mAP@0.50      : {metrics.box.map50 * 100:.2f}%')
print(f'Box mAP@0.50:0.95 : {metrics.box.map * 100:.2f}%')
print(f'Kecepatan Inferensi: {metrics.speed["inference"]:.2f} ms / citra')
print('=' * 75)


In [ ]:
# 6. Tampilkan Grafik Hasil Evaluasi
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

train_run_dir = "/content/runs/segment/egg_yolo26n_seg"

fig_files = [
    ('results.png', 'Grafik Loss, mAP Box & Mask tiap Epoch'),
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('MaskPR_curve.png', 'Precision-Recall Curve (Mask Segmentasi)'),
    ('BoxPR_curve.png', 'Precision-Recall Curve (Bounding Box)'),
]

for fname, title in fig_files:
    fpath = os.path.join(train_run_dir, fname)
    if os.path.exists(fpath):
        plt.figure(figsize=(10, 6))
        img = mpimg.imread(fpath)
        plt.imshow(img)
        plt.axis('off')
        plt.title(title, fontsize=14, fontweight='bold')
        plt.show()


In [ ]:
# 7. 📦 Kompres Semua Grafik & Unduh Otomatis 'best.pt' serta Grafik Evaluasi
import zipfile
from google.colab import files

train_run_dir = "/content/runs/segment/egg_yolo26n_seg"
best_weight_path = os.path.join(train_run_dir, "weights", "best.pt")
zip_output_path = "/content/runs/segment/yolo26_evaluation_graphs.zip"

# Kumpulkan semua file grafik & CSV
with zipfile.ZipFile(zip_output_path, 'w') as zf:
    for root, dirs, files_in_dir in os.walk(train_run_dir):
        for f in files_in_dir:
            if f.endswith(('.png', '.jpg', '.csv', '.yaml')):
                fpath = os.path.join(root, f)
                rel_path = os.path.relpath(fpath, train_run_dir)
                zf.write(fpath, arcname=rel_path)

print('\n' + '=' * 75)
print('📥 MENGUNDUH FILE KE KOMPUTER ANDA...')
print('=' * 75)

# 1. Download bobot best.pt
if os.path.exists(best_weight_path):
    print('-> Mengunduh weights/best.pt...')
    files.download(best_weight_path)
else:
    print('⚠️ File best.pt tidak ditemukan.')

# 2. Download paket grafik evaluasi
if os.path.exists(zip_output_path):
    print('-> Mengunduh yolo26_evaluation_graphs.zip...')
    files.download(zip_output_path)
else:
    print('⚠️ File zip grafik tidak ditemukan.')

print('✅ Seluruh proses selesai! Simpan best.pt ke folder yolov26/weights/best.pt di komputer Anda.')
